In [ ]:
import numpy as np
from random import randint, choice, random

# The number of piles is 3


# max number of items per pile
ITEMS_MX = 10

# Initialize starting position
def init_game()->list:
    return [randint(1,ITEMS_MX), randint(1,ITEMS_MX), randint(1,ITEMS_MX)]

# Based on X-oring the item counts in piles - mathematical solution
def nim_guru(_st:list)->(int,int):
    xored = _st[0] ^ _st[1] ^ _st[2]
    if xored == 0:
        return nim_random(_st)
    for pile in range(3):
        s = _st[pile] ^ xored
        if s <= _st[pile]:
            return _st[pile]-s, pile

# Random Nim player
def nim_random(_st:list)->(int,int):
    pile = choice([i for i in range(3) if _st[i]>0])  # find the non-empty piles
    return randint(1, _st[pile]), pile  # random move

In [2]:
def nim_qlearner(_st:list)->(int,int):
    global qtable
    # pick the best rewarding move, equation 1
    a = np.argmax(qtable[_st[0], _st[1], _st[2]])  # exploitation
    # index is based on move, pile
    move, pile = a%ITEMS_MX+1, a//ITEMS_MX
    # check if qtable has generated a random but game illegal move - we have not explored there yet
    if move <= 0 or _st[pile] < move:
        move, pile = nim_random(_st)  # exploration
    return move, pile  # action

In [3]:
Engines = {'Random':nim_random, 'Guru':nim_guru, 'Qlearner':nim_qlearner}

def game(_a:str, _b:str):
    state, side = init_game(), 'A'
    while True:
        engine = Engines[_a] if side == 'A' else Engines[_b]
        move, pile = engine(state)
        # print(state, move, pile)  # debug purposes
        state[pile] -= move
        if state == [0, 0, 0]:  # game ends
            return side  # winning side
        side = 'B' if side == 'A' else 'A'  # switch sides

def play_games(_n:int, _a:str, _b:str)->(int,int):
    from collections import defaultdict
    wins = defaultdict(int)
    for _ in range(_n):
        wins[game(_a, _b)] += 1
    # info
    print(f"{_n} games, {_a:>8s}{wins['A']:5d}  {_b:>8s}{wins['B']:5d}")
    return wins['A'], wins['B']

In [4]:
# Play games
play_games(1000, 'Random', 'Random')
play_games(1000, 'Guru', 'Random')
play_games(1000, 'Random', 'Guru')
play_games(1000, 'Guru', 'Guru') ;

1000 games,   Random  503    Random  497
1000 games,     Guru 1000    Random    0
1000 games,   Random    4      Guru  996
1000 games,     Guru  942      Guru   58


In [5]:
qtable, Alpha, Gamma, Reward = None, 1.0, 0.8, 100.0

# learn from _n games, randomly played to explore the possible states
def nim_qlearn(_n:int):
    global qtable
    # based on max items per pile
    qtable = np.zeros((ITEMS_MX+1, ITEMS_MX+1, ITEMS_MX+1, ITEMS_MX*3), dtype=np.float32)
    # play _n games
    for _ in range(_n):
        # first state is starting position
        st1 = init_game()
        while True:  # while game not finished
            # make a random move - exploration
            move, pile = nim_random(st1)
            st2 = list(st1)
            # make the move
            st2[pile] -= move  # --> last move I made
            if st2 == [0, 0, 0]:  # game ends
                qtable_update(Reward, st1, move, pile, 0)  # I won
                break  # new game

            qtable_update(0, st1, move, pile, np.max(qtable[st2[0], st2[1], st2[2]]))
            
            # Switch sides for play and learning
            st1 = st2

# Equation 3 - update the qtable
def qtable_update(r:float, _st1:list, move:int, pile:int, q_future_best:float):
    a = pile*ITEMS_MX+move-1
    qtable[_st1[0], _st1[1], _st1[2], a] = Alpha * (r + Gamma * q_future_best)

In [6]:
nim_qlearn(2000) 

In [7]:
# Play games
play_games(1000, 'Qlearner', 'Random')
play_games(1000, 'Random', 'Qlearner')

play_games(1000, 'Random', 'Random') ;

1000 games, Qlearner  697    Random  303
1000 games,   Random  312  Qlearner  688
1000 games,   Random  536    Random  464


In [8]:
# See the training size effect
n_train = (3, 10, 100, 1000, 10000, 50000, 100000)
Wins = []
for n in n_train:
    nim_qlearn(n)
    wins_a, wins_b = play_games(1000, 'Qlearner', 'Random')
    Wins += [wins_a/(wins_a+wins_b)]

1000 games, Qlearner  531    Random  469
1000 games, Qlearner  552    Random  448
1000 games, Qlearner  717    Random  283
1000 games, Qlearner  707    Random  293
1000 games, Qlearner  695    Random  305
1000 games, Qlearner  720    Random  280
1000 games, Qlearner  699    Random  301


In [9]:
# Check the ratio of wins wrt to size of the reinforcement model training
print(Wins)

[0.531, 0.552, 0.717, 0.707, 0.695, 0.72, 0.699]


In [10]:
# Function to print the entire set of states
def qtable_log(_fn:str):
    with open(_fn, 'w') as fout:
        s = 'state'
        for a in range(ITEMS_MX*3):
            move, pile = a%ITEMS_MX+1, a//ITEMS_MX
            s += ',%02d_%01d' % (move,pile)
        print(s, file=fout)
        for i, j, k in [(i,j,k) for i in range(ITEMS_MX+1) for j in range(ITEMS_MX+1) for k in range(ITEMS_MX+1)]:
            s = '%02d_%02d_%02d' % (i,j,k)
            for a in range(ITEMS_MX*3):
                r = qtable[i, j, k, a]
                s += ',%.1f' % r
            print(s, file=fout)

qtable_log('qtable_debug.txt')

In [11]:
%%html
<style>
    table {margin-left: 0 !important;}
    p {font-family: verdana;}
    li {font-family: verdana;}
    div {font-size: 10pt;}
</style>
<!-- Display markdown tables left oriented in this notebook. -->

Question 1: Describe the environment in the Nim learning model:

The environment is everything that falls outside of the agent and the environment communicates with the agent to determine a "reward" based on actions. In the case of the Nim learning model, we have multiple players which are pitted against each other. There are three piles of objects which can have up to 10 objects in each pile. The rules dictate the players each take turns removing objects from a single pile and they can remove any number of objects. The winning condition is forcing the opponent to take the last object. 

The environemtn initialzes each game with a random number of objects in each pile and Player 1 goes first each time. The reward is communicated to the agent via the Q-table. In this environment the Q-table is only accessed by Q-learner.

Question 2: Describe the agent(s) in the Nim learning model.

There are three agents in this game, the Q-learner, the random player, and the guru. Each of them interact with the environment which does fit the criteria of an agent. All three agents observe the state of the game. However, the Q-learner is the closest to a true agent. Neither the random player or guru receive any feedback from the environment in terms of reward. Also the random player and guru do not alter their behavior, but they still count as an agent since they interact with the environment. 


Question 3: Describe the reward and penalty in the Nim model.

Based on the original code there is only a reward of 100 for a win and no penality for loss. There is no reward or penalty for intermediate steps either.

Question 4: How many possible states can there be in a Nim game with 10 items per pile and 3 piles? 

Since each pile may have a total of 11 states, 0 to 10 and we have 3 piles. The game ends with all piles being empty. So a total of 11 x 11 x 11 states = 1331 states

Question 5: How many possible unique actions for the first players first action are possible with 10 items per pile and 3 piles? 

There are a total of 30 possible unique actions for player 1.

Question 6: Can the learner beat the Guru? Why or why not?

No, the learner cannot win based on its decisions. It may win a game through luck of xored state at the initialization of the game, but that is not a result of any learning or choices the Q-learner may make during a game. This is a result of the Guru and the games starting conditions. We saw that in the Guru vs Guru matches there is a distinct advantage to being the first player and making the initial play, we see in the following code that since init_game initializes with randint, most of the time the xored value is not zero which is a winning conditioning for the Guru if it goes first. The initial game state is define by this code

def init_game()->list:
    return [randint(1,ITEMS_MX), randint(1,ITEMS_MX), randint(1,ITEMS_MX)]

def nim_guru(_st:list)->(int,int):
    xored = _st[0] ^ _st[1] ^ _st[2]
    if xored == 0:
        return nim_random(_st)

By definition if the initial state is xored == 0 then any move the Guru makes xored non-zero allows the other player to return xored back to 0. The final state of the game is 0 ^ 0 ^ 0 = 0 with all piles empty. If you can force someone to a state where the board xored == 0 then you can guarantee a win. In the case of the Guru this only happens if the game starts in that state. This is a rare occurrence since the game is initialized with randint so and there is a specific bitwise relationship that has to occur between the three piles in order for xored to be zero.

The information gained from the trials below assocaited with Question 6 reinforce this. No matter how much the Learner trains against the Guru it cannot meaningfully beat the Guru through it's play. We saw even if the Learner trained for 1E6 games with the Guru it did not significantly affect the results of the playing against the Guru.


Question 7:  Find a way to improve the provided Nim game learning model. (Hint: How about 
penalizing the losses? Hint: It is indeed possible to find a better solution, which improves 
the way Q-learning updates its Q-table). You must code a solution and also demonstrate 
the improvement by reporting its performance against players (Random, Guru). 
Do not put the Guru player’s operating code inside the learning module, as this would 
defeat the purpose of reinforcement learning. However, you may train your improved Q
learner by having it playing against a Guru; using those games as experience is legitimate 
reinforcement learning.

We can adjust this a number of ways to improve it. In it's original state there is no loss penalty, so the learner never learns from a loss only it's wins. Second we need to fix the Bellman equation from qtable_update. Also, I adjusted the Alpha parameter to 0.5 this should help learning by mixing what it already knows and the new estimated best move. I updated the qtable_update to penalize the losses and reward the wins. Also, the Learner can learn from the opponents losses as well in this case. Their losses update the qtable with the losing move so the learner can learn from those as well.

Previous best win rate against the Random player with original Q-learner after training for 100000 games: 72%
Current best win rate against Random player with Improved Q-learner and training against Guru for 100000 games: 78%

We cannot train the Learner enough in order to beat the Guru. We can improve playing against the Random player, but the Learner cannot beat the Guru as it plays mathematically perfect.


In [ ]:
def nim_egreedy(st:list, epsilon:float)->(int,int):
    if random.random() < epsilon:
        return nim_random(st)  # exploration
    else:
        a = np.argmax(qtable[st[0], st[1], st[2]])  # exploitation
        move, pile = a%ITEMS_MX+1, a//ITEMS_MX
        if move <= 0 or st[pile] < move: 
            return nim_random(st)  # exploration
        return move, pile  # action

In [ ]:
qtable, Alpha, Gamma, Reward, Penalty = None, 0.5, 0.8, 100.0, -100.0

# We adjust the learner here by introducing the penalty for loosing and also by including the current q value and the learning rate in the update
# to make sure the value is updated based on the previous value and the future best value, not just the future best value
def nim_qlearn_2(_n:int, opponent='Random'):
    global qtable
    # based on max items per pile
    qtable = np.zeros((ITEMS_MX+1, ITEMS_MX+1, ITEMS_MX+1, ITEMS_MX*3), dtype=np.float32)

    opponent_fn = nim_guru if opponent == 'Guru' else nim_random
    # play _n games
    for _ in range(_n):
        # first state is starting position
        st1 = init_game()
        #define last move so we can update qtable with penalty for losses
        last = {'learner':None, 'opponent':None}
        turn = 'learner'
        while True:  # while game not finished
            if turn == 'learner':
                move, pile = nim_egreedy(st1, epsilon) # pick the best rewarding move
                st2 = list(st1) # copy the state
                st2[pile] -= move #> last move I made

                if st2 == [0, 0, 0]:  # game ends
                    qtable_update_2(Reward, st1, move, pile, 0)  # I won, update qtable with reward for current move
                    if last['opponent'] is not None:
                        qtable_update_2(Penalty, last['opponent']['state'], last['opponent']['move'], last['opponent']['pile'], 0)  # opponent lost
                    break  # new game

                qtable_update_2(0, st1, move, pile, np.max(qtable[st2[0], st2[1], st2[2]])) # update qtable with reward for current move and future best value
                last['learner'] = {'state': list(st1), 'move': move, 'pile': pile} # save the last move for the learner to update with penalty if opponent wins
                st1 = st2 # Switch sides for play and learning
                turn = 'opponent' # Switch to opponent's turn
            else:
                move, pile = opponent_fn(st1) # opponent makes a move
                st2 = list(st1) # copy the state
                st2[pile] -= move #> last move opponent made

                if st2 == [0, 0, 0]:  # game ends, opponent won
                    if last['learner'] is not None:
                        qtable_update_2(Penalty, last['learner']['state'], last['learner']['move'], last['learner']['pile'], 0)  # learner lost
                    break  # new game
                last['opponent'] = {'state': list(st1), 'move': move, 'pile': pile} 
                st1 = st2 # Switch sides for play and learning
                turn = 'learner' # Switch back learner's turn

            


# Updated the qtable update to include the current q value and the learning rate, also to make sure the value is updated based 
# on the previous value and the future best value, not just the future best value
def qtable_update_2(r:float, _st1:list, move:int, pile:int, q_future_best:float):
    a = pile*ITEMS_MX+move-1
    current_q = qtable[_st1[0], _st1[1], _st1[2], a]
    qtable[_st1[0], _st1[1], _st1[2], a] = current_q + Alpha * (r + Gamma * q_future_best - current_q)

In [13]:
#now we train the Q-learner by playing with the guru
nim_qlearn_2(2000, opponent='Guru')

In [14]:
# Play games again with the new trained model and compare to last results
play_games(1000, 'Qlearner', 'Random')
play_games(1000, 'Random', 'Qlearner')

play_games(1000, 'Random', 'Random') ;

1000 games, Qlearner  641    Random  359
1000 games,   Random  383  Qlearner  617
1000 games,   Random  509    Random  491


In [15]:
#Lets see the effect of training with guru for increasing number of games and then playing against random
n_train = (3, 10, 100, 1000, 10000, 50000, 100000)
Wins = []
for n in n_train:
    nim_qlearn_2(n, opponent='Guru')
    wins_a, wins_b = play_games(1000, 'Qlearner', 'Random')
    Wins += [wins_a/(wins_a+wins_b)]
print(Wins)

1000 games, Qlearner  489    Random  511
1000 games, Qlearner  481    Random  519
1000 games, Qlearner  502    Random  498
1000 games, Qlearner  595    Random  405
1000 games, Qlearner  672    Random  328
1000 games, Qlearner  756    Random  244
1000 games, Qlearner  778    Random  222
[0.489, 0.481, 0.502, 0.595, 0.672, 0.756, 0.778]


In [16]:
n_train = (3, 10, 100, 1000, 10000, 50000, 100000)
Wins = []
for n in n_train:
    nim_qlearn_2(n, opponent='Guru')
    wins_a, wins_b = play_games(1000, 'Qlearner', 'Guru')
    Wins += [wins_a/(wins_a+wins_b)]
print(Wins)

1000 games, Qlearner    2      Guru  998
1000 games, Qlearner    2      Guru  998
1000 games, Qlearner   10      Guru  990
1000 games, Qlearner    7      Guru  993
1000 games, Qlearner    3      Guru  997
1000 games, Qlearner    7      Guru  993
1000 games, Qlearner   10      Guru  990
[0.002, 0.002, 0.01, 0.007, 0.003, 0.007, 0.01]


In [17]:
n_train = (1000, 10000, 50000, 100000, 200000, 500000, 1E6) #lets increase the number of training games and see if we can match the guru's performance
Wins = []
for n in n_train:
    nim_qlearn_2(int(n), opponent='Guru')
    wins_a, wins_b = play_games(1000, 'Qlearner', 'Guru')
    Wins += [wins_a/(wins_a+wins_b)]
print(Wins)

1000 games, Qlearner    2      Guru  998
1000 games, Qlearner    3      Guru  997
1000 games, Qlearner    8      Guru  992
1000 games, Qlearner    8      Guru  992
1000 games, Qlearner    4      Guru  996
1000 games, Qlearner    4      Guru  996
1000 games, Qlearner    2      Guru  998
[0.002, 0.003, 0.008, 0.008, 0.004, 0.004, 0.002]
